In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import numpy as np

In [66]:
def make_sequences(data, window_size=7, output_steps=3): # window_size = hoe ver in de toekomst wil je voorspellen
    X = []
    for i in range(len(data) - window_size - output_steps + 1):
        X.append(data[i:i+window_size])
    return np.array(X)

In [67]:
def make_targets(data, window_size=7, output_steps=3): # window_size = hoe ver in de toekomst wil je voorspellen
    y = []
    for i in range(len(data) - window_size - output_steps + 1):
        y.append(data[i+window_size:i+window_size+output_steps])
    return np.array(y)

In [68]:
def make_dataset(train_dfs):
    X_datasets = []
    y_datasets = []
    for train_df in train_dfs:
        dfy = train_df[["periodic"]].values
        dfx = train_df.drop(columns=["poi_id", "periodic"]).values
        X_train = make_sequences(dfx)
        y_train = make_targets(dfy)

        X_datasets.append(X_train)
        y_datasets.append(y_train)

    X_dataset = X_datasets[0]
    for ds in X_datasets[1:]:
        X_dataset = np.concatenate([X_dataset, ds])

    y_dataset = y_datasets[0]
    for ds in y_datasets[1:]:
        y_dataset = np.concatenate([y_dataset, ds])
    
    return X_dataset, y_dataset

In [ ]:
# 1 laad de dataset in
np.random.seed(42)
timesteps = 100
t = np.arange(timesteps)

df = pd.DataFrame({
    "poi_id": ["10"]*20 + ["20"]*20 + ["30"]*20 + ["40"]*20 + ["50"]*20,
    "sin": np.sin(0.1 * t),
    "cos": np.cos(0.1 * t),
    "trend": 0.01 * t,
    "noise": np.random.randn(timesteps) * 0.1,
    "periodic": np.sin(0.05 * t) + 0.5,
    "category": np.random.choice(["A", "B", "C"], size=timesteps),
})

# 2 maak de standardscaler en hotencoding pipeline
categorical_cols = ["category", "poi_id"]
numerical_cols = [col for col in df.columns if col not in categorical_cols]
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
    ]
)

# 3 verdeel je dataset in train_df, val_df en test_df
temp_df = df[:85]
train_df = temp_df[:70]
val_df = temp_df[70:]
test_df = df[85:]

# 4 hotencode en standardscale de dfs
X_train = preprocessor.fit_transform(train_df)
X_val = preprocessor.transform(val_df)
X_test = preprocessor.transform(test_df)

# 5 maak een kopie van de target kolom
target_kolom = df[["poi_id", "periodic"]].copy()

# 6 verdeel je gekopieerde kolom in y_train, y_val en y_test
y_temp_df = target_kolom[:85]
y_train = y_temp_df[:70] # eerste 70
y_val = y_temp_df[70:] # van 70-85
y_test = target_kolom[85:] # laatste 15

# maak X_train een dataset en voeg X_train en y_train samen -> dit ook van val en test
all_cols = preprocessor.get_feature_names_out()
df_X_train_transformed = pd.DataFrame(X_train, columns=all_cols)
y_train = y_train.reset_index(drop=True)
df_X_train_transformed = df_X_train_transformed.reset_index(drop=True)
concat_train = pd.concat([y_train, df_X_train_transformed], axis=1)

df_X_val_transformed = pd.DataFrame(X_val, columns=all_cols)
y_val = y_val.reset_index(drop=True)
df_X_val_transformed = df_X_val_transformed.reset_index(drop=True)
concat_val = pd.concat([y_val, df_X_val_transformed], axis=1)

df_X_test_transformed = pd.DataFrame(X_test, columns=all_cols)
y_test = y_test.reset_index(drop=True)
df_X_test_transformed = df_X_test_transformed.reset_index(drop=True)
concat_test = pd.concat([y_test, df_X_test_transformed], axis=1)

# splits de dataset per poi_id
train_dfs = [group for _, group in concat_train.groupby("poi_id")]
val_dfs = [group for _, group in concat_val.groupby("poi_id")]
test_dfs = [group for _, group in concat_test.groupby("poi_id")]

X_train, y_train = make_dataset(train_dfs)
X_val, y_val = make_dataset(val_dfs)
X_test, y_test = make_dataset(test_dfs)

je moet ervoor zorgen dat er per dataset per poi minstens 1 tijdreeks gemaakt kan worden. anders geeft hij een dimension error

-> elke dataset per poi moet minstens: timesteps om mee te voorspellen + aantal tijdstappen in de toekomst voorspellen